# 데이터 전처리 & 데이터 증강

In [36]:
import matplotlib.pyplot as plt
from torchvision.transforms import ToTensor
from torchvision.datasets import CIFAR10
import torch

In [ ]:
# 데이터셋 불러오기
# training_dataset =  CIFAR10(
#    root = './',
#    train = True,
#    download = True,
#    transform=ToTensor()
#)
#test_dataset =  CIFAR10(
#    root = './',
#    train = False,
#    download = True,
#    transform=ToTensor()
#)  
#
#for i in range(9):
#    img, label = training_dataset[i]
#    plt.subplot(3,3,i+1)
#    plt.imshow(img.permute(1,2,0))  #(C,H,W):0 1 2 -- 1 2 0   (H,W,C)
#plt.show() 

In [ ]:
# 크롭핑, 뒤집기 추가
import torchvision.transforms as T
from torchvision.transforms import Compose
from torchvision.transforms import RandomHorizontalFlip, RandomCrop


transform = Compose([
    T.ToTensor(),
    RandomCrop( (32,32), padding=4),
    RandomHorizontalFlip(p=0.5)
])


training_dataset =  CIFAR10(
    root = './',
    train = True,
    download = True,
    transform=transform
)
test_dataset =  CIFAR10(
    root = './',
    train = False,
    download = True,
    transform=transform
)  


for i in range(9):
    img, label = training_dataset[i]
    plt.subplot(3,3,i+1)
    plt.imshow(img.permute(1,2,0))  #(C,H,W):0 1 2 -- 1 2 0   (H,W,C)
plt.show() 

## 정규화
- 정규화 하는 이유: 채널 정규화

In [ ]:
# 데이터 전처리에 '정규화' 추가하기
from torchvision.transforms import Normalize

transform = Compose([
    T.ToPILImage(),    
    RandomCrop( (32,32), padding=4),
    RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    # 데이터 정규화
    Normalize((0.5,0.5,0.5),(0.2,0.2,0.2)),
    T.ToPILImage(),    
])
training_dataset =  CIFAR10(
    root = './',
    train = True,
    download = True,
    transform=transform
)
test_dataset =  CIFAR10(
    root = './',
    train = False,
    download = True,
    transform=transform
)  
for i in range(9):
    # img, label = training_dataset[i]
    plt.subplot(3,3,i+1)
    # plt.imshow(img.permute(1,2,0))  #(C,H,W):0 1 2 -- 1 2 0   (H,W,C)
    plt.imshow(transform(training_dataset.data[i]))
plt.show() 

# 딥러닝

In [ ]:
import torch.nn as nn

class BasicBlock(nn.Module):
    
    def __init__(self, int_channels, out_channels, hidden_dim):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(int_channels, hidden_dim, kernel_size=3,padding=1)
        self.conv2 = nn.Conv2d(hidden_dim, out_channels,kernel_size=3,padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
    
    def forward(self, x):
        x = self.relu( self.conv1(x) )
        x = self.relu( self.conv2(x) )
        out = self.pool(x)
        return out

In [ ]:
class CNN(nn.Module):

    def __init__(self, num_class):
        super(CNN, self).__init__( )
        self.block1 = BasicBlock(3,64,64)
        self.block2 = BasicBlock(64,128,128) #in: 위의 out과 맞추기
        
        # 분류기
        self.fc1 = nn.Linear(128*8*8, 2048) #in 직접 계산해줘야 함.
        self.fc2 = nn.Linear(2048, 256)
        self.fc3 = nn.Linear(256, num_class)
        self.relu = nn.ReLU() #기울기 소실 방지!
    
    def forward(self, x):
        x = self.block1(x)
        x= self.block2(x) 
        # (-1, 128*8*8)
        x = torch.flatten(x, start_dim=1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        out = self.fc3(x)
        return out

In [ ]:
# 샘플 데이터로 '구조 확인'해보기
X = torch.randn(4,3,32,32)
model = CNN(10)
model(X)

In [ ]:
from torchsummary import summary
model = CNN(10)
summary(model, (3,32,32))

In [ ]:
for name, param in model.named_parameters():
    print(f'{name} {list(param.shape)}')

In [ ]:
model.fc3
# 클래스 수 바꾸기
# model.fc3 = nn.Linear(in_features=256, out_features=2, bias=True)

In [40]:
import torchvision.transforms as T
from torchvision.transforms import Compose
from torchvision.transforms import RandomHorizontalFlip, RandomCrop, Normalize
from torch.optim import Adam
from torch.utils.data.dataloader import DataLoader
from tqdm import tqdm

# 1. 데이터 증강
transform = Compose([
    RandomCrop( (32,32), padding=4),
    RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    Normalize((0.5,0.5,0.5),(0.2,0.2,0.2)),
])

# 데이터셋
training_dataset =  CIFAR10(root = './',train = True,download = True, transform=transform)
test_dataset =  CIFAR10(root = './',train = False,download = True, transform=transform) 

# 데이터 로더
train_loader = DataLoader(training_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNN(10)
model.to(device)

#---- 2. 학습루프 ----
lr = 1e-3
optim = Adam(model.parameters(), lr=lr)
epochs = 1

for epoch in range(epochs):
    for data, label in train_loader:
        optim.zero_grad()
        preds = model(data.to(device))
        loss = nn.CrossEntropyLoss()(preds,label.to(device))
        loss.backward()
        optim.step()
    if (epoch+1) % 10 == 0:
        print(f'epoch: {epoch+1} | loss: {loss.item():.4f}')


torch.save(model.state_dict(), 'cifar.pth') #가중치+모델저장 or 가중치만 저장

In [42]:
# 평가
model.load_state_dict(torch.load('cifar.pth', map_location=device))

# 예측
num_corr = 0

with torch.no_grad():
    for data, label in test_loader:
        output = model(data.to(device))
        preds = output.data.max(1)[1]
        corr = preds.eq(label.to(device).data).sum().item()
    print(f'Accuracy: {num_corr / len(test_loader)}')

Accuracy: 0.0
